## Importing libraries

In [ ]:
import foldersfiles as ff
import functions as fn

import pickle

import numpy as np
import pandas as pd

import json

from pathlib import Path

## Importing json file to a proper python dictionary

In [ ]:
# Dictionary containing all the json files imported (json_file_name : json_content)
json_files_dict = {}

for path in sorted(Path(str(ff.RAW_DATA_DIR / 'dati_eumaco_con_pseudo')).glob('*.json')):
    try:
        filename = path.stem
        json_content = json.loads(path.read_text()) # Closes automatically the file
        json_files_dict[filename] = json_content
    except json.JSONDecodeError:
        print(f'Skipping invalid JSON: {path}')

# List of all the json files names
json_filenames = list(json_files_dict)

## Make a pretty-print version of all the jsons

In [ ]:
# input_dir = ff.RAW_DATA_DIR / 'dati_eumaco_con_pseudo'

# output_dir = ff.PROCESSED_DATA_DIR / ff.JSONST
# fn.pretty_print_folder(input_dir, output_dir, indent='\t')

# output_dir = ff.PROCESSED_DATA_DIR / ff.JSONS2
# fn.pretty_print_folder(input_dir, output_dir, indent=2)

# output_dir = ff.PROCESSED_DATA_DIR / ff.JSONS4
# fn.pretty_print_folder(input_dir, output_dir, indent=4)

## To pandas df

In [ ]:
date_fields = [
    'assignment.start_at' ,
    'assignment.end_at' ,
    'assignment.created_at' ,
    'assignment.updated_at' ,

    'created_at' ,
    'updated_at' ,
    'registration_time' ,

    'transmission_data.timestamp' ,
    'transmission_data.time.referenceTimeStamp' ,
]

### Lists of fields to import

In [ ]:
external_fields = [
    'patient_cf' ,
    'assignment_id' ,
    # 'archivied' ,
    # 'completed' ,

    # 'assignment.id' ,
    # 'assignment.start_at' ,
    # 'assignment.end_at' ,
    # 'assignment.active' ,
    # 'assignment.created_at' ,
    # 'assignment.updated_at' ,
    # 'assignment.hub_sn' ,
    # 'assignment.completed' ,
]

In [ ]:
# Common to every transmission type

instance_common_fields = [
    'id' ,
    # 'type' ,
    # 'report' ,
    # 'created_at' ,
    # 'updated_at' ,
    # 'registration_time' ,
    # 'device_sn' ,

    # # transmission_data
        # 'transmission_data.device' ,
        # 'transmission_data.serialnumber' ,
        'transmission_data.timestamp' ,
]

In [ ]:
# Specific (by key) for every instance

transm_specific_fields = {

'BloodPressureTransmission' : [
    # # systolic
        'transmission_data.systolic.value' ,
        # 'transmission_data.systolic.unit' ,
    # # diastolic
        'transmission_data.diastolic.value' ,
        # 'transmission_data.diastolic.unit' ,
    # # map
        'transmission_data.map.value' ,
        # 'transmission_data.map.unit' ,
    # # heartrate
        'transmission_data.heartrate.value' ,
        # 'transmission_data.heartrate.unit' ,
    # # battery
        # 'transmission_data.battery.value' ,
        # 'transmission_data.battery.unit' ,
],


'SpO2Transmission' : [
    # # battery
        # 'transmission_data.battery.value' ,
        # 'transmission_data.battery.unit' ,
    # # spo2
        'transmission_data.spo2.value' ,
        # 'transmission_data.spo2.unit' ,
    # # heartrate
        'transmission_data.heartrate.value' ,
        # 'transmission_data.heartrate.unit' ,
],


'WeightTransmission' : [
    # # time
        # 'transmission_data.time.referenceTimeStamp' ,
        # 'transmission_data.time.offset' ,
        # 'transmission_data.time.unit' ,
    # # weight
        'transmission_data.weight.value' ,
        # 'transmission_data.weight.unit' ,
],

}

### Actual transformation - CLICK HERE

In [ ]:
transm_field = 'BloodPressureTransmission'
# transm_field = 'SpO2Transmission'
# transm_field = 'WeightTransmission'

In [ ]:
letters = 4

In [ ]:
def load_transmission_df(transmission):
    '''
    Helper function to apply fn.json_to_df easier on a loop
    '''
    data = fn.json_to_df(
        json_files_dict, json_filenames, date_fields,
        transm_field = transmission,
        external_json_fields = external_fields,
        instance_common_fields = instance_common_fields,
        transm_specific_fields = transm_specific_fields[transmission],
        letters=letters,
    )
    return pd.DataFrame(data)

In [ ]:
df = load_transmission_df(transm_field)

In [ ]:
df.info()

In [ ]:
df

## Analysis and overview

### Fields `assignment_id` and `assignment:id` are always equal to the filename? -> yes

In [ ]:
not_equal = []
for json_name in json_filenames:
    assignment_id = json_files_dict[json_name]['assignment_id']
    assignment_id_deep = json_files_dict[json_name]['assignment']['id']
    if int(json_name) == assignment_id == assignment_id_deep:
        continue
    else:
        not_equal.append(json_name)

print(f"Number of jsons where ('json filename' != 'assignment_id' != 'assignment:id'): {len(not_equal)}")

### Check if all the "unit" fields inside `transmission_data` fields has the same value

In [ ]:
unit_columns_to_check = {
    'BloodPressureTransmission':[
        'tran.syst.unit' ,
        'tran.dias.unit' ,
        'tran.map.unit' ,
        'tran.hear.unit' ,
        'tran.batt.unit' ,
    ],
    'SpO2Transmission':[
        'tran.batt.unit' ,
        'tran.spo2.unit' ,
        'tran.hear.unit' ,
    ],
    'WeightTransmission':[
        'tran.time.unit' ,
        'tran.weig.unit' ,
    ]
}

In [ ]:
columns = unit_columns_to_check[transm_field]

for column in columns:
    if df[column].nunique(dropna=False) == 1:
        print(f'{column}:\tsingle value ({df[column].iloc[0]})')
    else:
        raise Exception(f'column "{column}" has multiple values')

### Time fields

How many instances of the internal fields `registration_time` and `tran.timestamp` has different values?

In [ ]:
different = df['registration_time'] != df['tran.timestamp']

print(different.sum())

#### External time fields

These external time fields are analyzed directly from the json because they are found one time for every json file.

Returns a dictionary of the external fields like `{json_name: {time_field1 : value, ...}, ...}`

In [ ]:
# Returns a dict where:
# - key is the json file name
# - value is a dict where:
#   * key is the external time field
#   * value is the date value of the field

# sample:
# {json_filename: {
    # 'start_at': date,
    # 'end_at': date,
    # 'created_at': date,
    # 'updated_at': date,
# }}

external_time_values = {}
external_time_fields = ['start_at', 'end_at', 'created_at', 'updated_at']

for json_name in json_filenames:
    external_time_values[json_name] = {}

    for field in external_time_fields:
        value = pd.Timestamp(json_files_dict[json_name]['assignment'][field])
        external_time_values[json_name][field] = value

external_time_values['31636']

`start_at` is always lower than the others

In [ ]:
# Proves that the field start_at
# always holds the earliest external date value

for json in external_time_values:
    values = external_time_values[json]
    if not(values['start_at'] < min(
        values['end_at'],
        values['created_at'],
        values['updated_at'],
    )):
        print(json)

Returns a dictionary where of the minimum and maximum values of the external fields `{json_name : {'min': minimum, 'max': maximum}, ...}`

In [ ]:
# Returns a dict where:
# - key is the json file name
# - value is a dict where:
#   * key is 'min' or 'max'
#   * value is lowest or highest date value among the external time fields

# sample:
# {json_filename: {
    # 'min': date,
    # 'max': date,
# }}

ext_minmax = {}
for json in external_time_values:
    values = external_time_values[json]
    minimum = values['start_at']
    maximum = values[max(values, key=values.get)]

    # ext_minmax[json] = [minimum, maximum] # Variant for list
    ext_minmax[json] = {'min': minimum, 'max': maximum}

ext_minmax['31636']

In [ ]:
def internal_btw_external(json_files_dict, json_filenames, transm_to_check, ext_minmax, int_mid_timefield, *, ids=False):
    '''
    Takes in input:
    - a dictionary {json_filename: json_file_content}
    - a list of json filenames
    - the transmission subfield to check
    - a dictionary with all the minimum and maximum dates for every json file
    - an "internal" timefield to check

    Check if the int_mid_timefield stays between min and max dates for every json file

    Returns an overview dictionary {'problems': {json_file: [position, ...]}, 'missing': [json_files]} that tells:
    - given a json file which have the field different from the sample one:
        - the instance's position within the file, if instance == False (default behaviour)
        - the instance's if field, if ids == True
        - the actual instance structure, if instance == True
    - which json files misses the transmission subfield to check
    '''

    # midtime = ['created_at', 'updated_at', 'registration_time', 'timestamp']

    # The returned object will have this structure:
    # {'different': {json_file: [position, ...]}, 'missing': [json_files]}
    overview = {'problems': {}, 'missing': []}

    # Iterates over the jsons
    for json_name in json_filenames:

        # transmissions will be a dict that contains the actual "transmissions" field of the json
        # {transm_to_check: content}
        transmissions = json_files_dict[json_name].get('transmissions', {})

        # If the transmission field to check is NOT present:
        # Append the json name to the missing list ...
        # .. and go directly to the next json file
        if transm_to_check not in transmissions:
            overview['missing'].append(json_name)
            continue

        # If the transmission field to check is present in the json:

        # Place in instances the complete list of instances of the transm_to_check
        instances = transmissions[transm_to_check][0]

        # Iterate over the long list of the instances and set:
        # - t_position as the transmission field instance's position
        # - t_instance as the the transmission field's instance itself
        for t_position, t_instance in enumerate(instances):

            if int_mid_timefield == 'timestamp':
                midtime = t_instance['transmission_data'][int_mid_timefield]
            else:
                midtime = t_instance[int_mid_timefield]

            condition = (
                ext_minmax[json_name]['min'] <
                # pd.Timestamp(t_instance['transmission_data']['timestamp']) <
                pd.Timestamp(midtime) <
                ext_minmax[json_name]['max']
            )
            if condition == False:
                if ids == False:
                    overview['problems'].setdefault(json_name, []).append(t_position)
                else:
                    overview['problems'].setdefault(json_name, []).append(t_instance['id'])

    return overview

In [ ]:
internals = internal_btw_external(json_files_dict, json_filenames, transm_field, ext_minmax, 'timestamp')
internals

### Check if the "external" fields except the `transmissions` have consistent values

#### booleans

##### assignment_ids

In [ ]:
column1 = 'assignment_id'
column2 = 'assignment.id'
print(fn.columns_consistency(df, column1, column2, bidirectional=True, return_number=True))

##### archived/completed

In [ ]:
column1 = 'archivied'
column2 = 'completed'

print(fn.columns_consistency(df, column1, column2, bidirectional=True, return_number=True))

##### completed/assignment:completed

In [ ]:
column1 = 'completed'
column2 = 'assignment.completed'

print(fn.columns_consistency(df, column1, column2, bidirectional=True, return_number=True))

##### assignment:active/assignment:completed

In [ ]:
column1 = 'assignment.active'
column2 = 'assignment.completed'

print(fn.columns_consistency(df, column1, column2, bidirectional=True, return_number=True))

##### archivied/assignment:completed

In [ ]:
column1 = 'archivied'
column2 = 'assignment.completed'

print(fn.columns_consistency(df, column1, column2, bidirectional=True, return_number=True))

## Parsing and reformatting

#### All the datetime fields has the same UTC timezone, so drop this information

Proof

In [ ]:
dt_cols = df.select_dtypes(include='datetimetz').columns

all_utc = True

for col in dt_cols:
    if str(df[col].dt.tz) != 'UTC':
        print(f'{col} is not UTC: {df[col].dt.tz}')
        all_utc = False

print('All the datetime column has UTC timezone?', all_utc)

Drop them

In [ ]:
def drop_utc_timezone(dataframe, dt_cols):
    '''
    Drops timezone information if all the timezone-aware datetime columns use UTC.
    If there are no timezone-aware datetime columns, or at least one of them does not use UTC, no timezone information is dropped.

    Takes in input the dataframe and the index of the columns to drop

    Returns a copy of the dataframe with the UTC timezone information removed
    '''
    dataframe = dataframe.copy()
    timezones = dataframe[dt_cols].apply(lambda col: str(col.dt.tz))

    if len(dt_cols) > 0 and timezones.nunique() == 1 and timezones.iloc[0] == 'UTC':
        for col in dt_cols:
            dataframe[col] = dataframe[col].dt.tz_localize(None)

    return dataframe

In [ ]:
df = drop_utc_timezone(df, dt_cols)

### Rename columns

In [ ]:
rename_dict = {

# # External fields

    'patient_cf'              : 'pseudopaz' ,
    'assignment_id'           : 'json_id' ,
    # 'archivied'               : '' ,
    # 'completed'               : '' ,

    # 'assignment.id'           : '' ,
    # 'assignment.start_at'     : '' ,
    # 'assignment.end_at'       : '' ,
    # 'assignment.active'       : '' ,
    # 'assignment.created_at'   : '' ,
    # 'assignment.updated_at'   : '' ,
    # 'assignment.hub_sn'       : '' ,
    # 'assignment.completed'    : '' ,


# # Common to every transmission type

    'id'                                   : 'transm_id' ,
    # 'type'                                 : '' ,
    # 'report'                               : '' ,
    # 'created_at'                           : '' ,
    # 'updated_at'                           : '' ,
    # 'registration_time'                    : '' ,
    # 'device_sn'                            : '' ,

    # # transmission_data
        # 'transmission_data.device'         : '' ,
        # 'transmission_data.serialnumber'   : '' ,
        'transmission_data.timestamp'      : 'datetime' ,


# # Specific for every instance

# # BloodPressureTransmission
    # # systolic
        'transmission_data.systolic.value'    : 'syst bp [mmHg]' ,
        # 'transmission_data.systolic.unit'     : '' ,
    # # diastolic
        'transmission_data.diastolic.value'   : 'diast bp [mmHg]' ,
        # 'transmission_data.diastolic.unit'    : '' ,
    # # map
        'transmission_data.map.value'         : 'map [mmHg]' ,
        # 'transmission_data.map.unit'          : '' ,
    # # heartrate
        # 'transmission_data.heartrate.value'   : '' ,   # Duplicate field! See below
        # 'transmission_data.heartrate.unit'    : '' ,
    # # battery
        # 'transmission_data.battery.value'     : '' ,
        # 'transmission_data.battery.unit'      : '' ,

# # SpO2Transmission
    # # battery
        # 'transmission_data.battery.value'     : '' ,
        # 'transmission_data.battery.unit'      : '' ,
    # # spo2
        'transmission_data.spo2.value'        : 'spo2 [%]' ,
        # 'transmission_data.spo2.unit'         : '' ,
    # # heartrate
        # 'transmission_data.heartrate.value'   : '' ,   # Duplicate field! See below
        # 'transmission_data.heartrate.unit'    : '' ,

# # WeightTransmission
    # # time
        # 'transmission_data.time.referenceTimeStamp'    : '' ,
        # 'transmission_data.time.offset'                : '' ,
        # 'transmission_data.time.unit'                  : '' ,
    # # weight
        'transmission_data.weight.value'               : 'weight [kg]' ,
        # 'transmission_data.weight.unit'                : '' ,

}

# BloodPressureTransmission and SpO2Transmission has an identical 'transmission_data.heartrate.value' field
# so the renaming process must take care of this
rename_dict_for_hr = {
    'BloodPressureTransmission': {
        'transmission_data.heartrate.value': 'bpt hr [bpm]',
    },
    'SpO2Transmission': {
        'transmission_data.heartrate.value': 'spo2 hr [bpm]',
    },
    'WeightTransmission': {},
}

In [ ]:
def rename_transmission_columns(dataframe, rename_dict, hr_renames, letters):
    '''
    Renames the columns using shared and transmission-specific labels.

    Takes in input:
    - the dataframe
    - rename_dict, a dictionary {field_path: label} containing the shared column labels
    - hr_renames, a dictionary {field_path: label} containing the transmission-specific heart rate labels,
        or an empty dictionary for transmissions without heart rate
    - letters: an int specifying the compression length for instance field names,
        or None if they are prefixed with 'inst.'

    Combines the two dictionaries, giving priority to hr_renames if a key is present in both.

    Returns a dataframe with the renamed columns, without modifying the input dataframe
    '''
    names = {**rename_dict, **hr_renames}
    column_mapping = {}

    for field, label in names.items():
        if field in external_fields:
            column_name = field
        elif letters is not None:
            column_name = fn.compress_field_path(field, letters=letters)
        else:
            column_name = f'inst.{field}'
        column_mapping[column_name] = label

    return dataframe.rename(columns=column_mapping)

In [ ]:
df = rename_transmission_columns(df, rename_dict, rename_dict_for_hr[transm_field], letters)

## Export

In [ ]:
# # General pickle
# with open(ff.PROCESSED_DATA_DIR / ff.PICKLES / f'{ff.EUMACO}-json_files_dict.pickle', 'wb') as file:
#     pickle.dump(json_files_dict, file, protocol=pickle.HIGHEST_PROTOCOL)

### Export only pseudopazs

In [ ]:
# pps_list = []

# # Iterates over the jsons
# for json_name in json_filenames:

#     # current_json is
#     current_json = json_files_dict[json_name]

#     # Appends to the list the pseudopaz
#     pps_list.append(current_json['patient_cf'])

# pps = pd.Series(pps_list)
# pps = pps.sort_values(ignore_index=True)

# # Pickle
# pps.to_pickle(ff.PROCESSED_DATA_DIR / ff.PICKLES / ff.PSEUDOPATIENTS / f'{ff.EUMACO}-pp.pickle')
# pps.drop_duplicates().to_pickle(ff.PROCESSED_DATA_DIR / ff.PICKLES / ff.PSEUDOPATIENTS / f'{ff.EUMACO}-pp-uniq.pickle')
# with open(ff.PROCESSED_DATA_DIR / ff.PICKLES / ff.PSEUDOPATIENTS / f'{ff.EUMACO}-pp-uniq-list.pickle', 'wb') as file:
#     pickle.dump(pps.drop_duplicates().tolist(), file, protocol=pickle.HIGHEST_PROTOCOL)

### Export only the dataset specfic to the transmission field

In [ ]:
transm_field_short = {
    'BloodPressureTransmission': 'bpt',
    'SpO2Transmission': 'spo2',
    'WeightTransmission': 'wt',
}

In [ ]:
# # Parquet
# df.to_parquet(ff.CLEANED_PROCESSED_DATA_DIR / ff.PARQUETS / f'{ff.EUMACO}-{transm_field_short[transm_field]}.parquet')

# # Pickle
# with open(ff.CLEANED_PROCESSED_DATA_DIR / ff.PICKLES / f'{ff.EUMACO}-{transm_field_short[transm_field]}.pickle', 'wb') as file:
#     pickle.dump(df, file, protocol=pickle.HIGHEST_PROTOCOL)

# # Excel
# df.to_excel(ff.CLEANED_PROCESSED_DATA_DIR / ff.EXCELS / f'{ff.EUMACO}-{transm_field_short[transm_field]}.xlsx',
#     sheet_name=f'{ff.EUMACO}-{transm_field_short[transm_field]}',
#     index=False
# )

### Export a single dataset for all the transmission fields (bpt, spo2, wt)

In [ ]:
def formatted_transmission_df(transmission):
    '''
    Creates a dataframe already reformatted (timezone info dropped and columns renamed)
        by using the proper helper functions load_transmission_df, drop_utc_timezone and rename_transmission_columns

    Takes in input:
    - the transmission fields to import ('BloodPressureTransmission', 'SpO2Transmission' or 'WeightTransmission')
        (useful to loop the process for all the three fields)

    Returns a dataframe ready for export or concatenation with the other transmission dataframes
    '''
    dataframe = load_transmission_df(transmission)
    dataframe = drop_utc_timezone(dataframe, dt_cols)
    return rename_transmission_columns(dataframe, rename_dict, rename_dict_for_hr[transmission], letters)

In [ ]:
dataframes = []
for transmission in transm_field_short:
    dataframes.append(formatted_transmission_df(transmission))

df_all = (
    pd.concat(dataframes, ignore_index=True)
    .assign(datetime=lambda df: pd.to_datetime(df["datetime"]))
    .sort_values(["pseudopaz", "datetime"])
    .reset_index(drop=True)
)

In [ ]:
# Parquet
df_all.to_parquet(ff.CLEANED_PROCESSED_DATA_DIR / ff.PARQUETS / f'{ff.EUMACO}-all.parquet')

# Pickle
with open(ff.CLEANED_PROCESSED_DATA_DIR / ff.PICKLES / f'{ff.EUMACO}-all.pickle', 'wb') as file:
    pickle.dump(df_all, file, protocol=pickle.HIGHEST_PROTOCOL)

# Excel
df_all.to_excel(ff.CLEANED_PROCESSED_DATA_DIR / ff.EXCELS / f'{ff.EUMACO}-all.xlsx',
    sheet_name='all',
    index=False
)